# AI Governance and Responsible AI: Hands-on Guide with Gemini

AI Governance and Responsible AI (RAI) ensure that AI technologies are developed and deployed in a manner that is safe, ethical, transparent, and aligned with human values.

### Key Pillars of Responsible AI:
1. **Safety & Policy Alignment**: Preventing the generation of harmful, toxic, or dangerous content.
2. **Fairness & Bias Mitigation**: Minimizing stereotypical or discriminatory behaviors.
3. **Transparency & Explainability**: Understanding why and how models make decisions.
4. **Robustness & Predictability**: Ensuring the outputs are structured and reliable enough for downstream automated verification.

In this notebook, we'll demonstrate how to enforce these practices using the **Google GenAI SDK** and the `gemini-2.5-flash` model.

## 1. Environment Setup

First, install the modern `google-genai` library and initialize the Gemini client. Make sure you have added your API key to Colab's Secrets manager under the key `GOOGLE_API_KEY`.

In [ ]:
!pip --quiet install google-genai pydantic -q

In [ ]:
from google import genai
from google.genai import types
from google.colab import userdata

# Initialize the client securely using Colab Secrets
try:
    api_key = userdata.get('gemini_key')
    client = genai.Client(api_key=api_key)
    print("Gemini Client initialized successfully.")
except Exception as e:
    print(f"Initialization error: {e}\nEnsure GOOGLE_API_KEY is configured in Colab Secrets.")

Gemini Client initialized successfully.


## 2. Enforcing Safety Thresholds

Gemini allows developers to adjust safety thresholds to block content that violates policies regarding harassment, hate speech, sexually explicit material, or dangerous content. This is a foundational pillar of **AI Governance**.

In [ ]:
def generate_with_safety_governance(prompt: str) -> str:
    """
    Generates content using Gemini while explicitly configuring safety settings
    to block potentially harmful material according to governance policies.
    """
    # Define strict safety boundaries
    safety_settings = [
        types.SafetySetting(
            category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
            threshold=types.HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
        ),
        types.SafetySetting(
            category=types.HarmCategory.HARM_CATEGORY_HARASSMENT,
            threshold=types.HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
        ),
        types.SafetySetting(
            category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
            threshold=types.HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
        ),
    ]

    config = types.GenerateContentConfig(
        safety_settings=safety_settings,
        temperature=0.2 # Lower temperature for more consistent and responsible outputs
    )

    try:
        response = client.models.generate_content(
            model='gemini-3.5-flash-lite',
            contents=prompt,
            config=config
        )
        return response.text
    except Exception as e:
        return f"Request blocked or failed due to governance safety policies: {e}"

# Test the governed execution
#test_prompt='some wrong content'

test_prompt = "Explain how to build a healthy culture of trust and ethical compliance in a modern engineering team."
print(generate_with_safety_governance(test_prompt))

Deploying an emotion detection and focus-monitoring AI system in an office environment involves significant technical, ethical, legal, and cultural challenges. While the stated goal is often to improve productivity or well-being, such systems frequently backfire and introduce severe risks.

Here is a comprehensive breakdown of the key considerations, challenges, and alternative approaches you should evaluate before proceeding.

---

### 1. Scientific and Technical Validity
* **The "Emotion AI" Fallacy:** The scientific consensus (including comprehensive reviews by the Association for Psychological Science) largely concludes that facial expressions alone are unreliable indicators of internal emotions, focus, or productivity. A person’s face does not universally reveal *how* they feel; it often reflects context, culture, and immediate physical actions rather than cognitive engagement.
* **False Positives and Bias:** Computer vision models are notoriously prone to bias. Factors such as sk

## 3. System Instructions for Alignment & Bias Mitigation

System instructions are highly effective for aligning model behavior. They force the model to adopt a neutral, objective, and unbiased persona, which is crucial for fair decision-making.

In [ ]:
def evaluate_bias_neutrality(user_scenario: str) -> str:
    """
    Instructs the LLM to act as an objective, unbiased governance auditor
    evaluating a workplace or organizational policy scenario.
    """
    system_instruction = (
        "You are an expert AI Governance Officer. Your role is to analyze scenarios "
        "for potential bias, lack of inclusivity, and ethical risks. Always maintain a neutral, "
        "constructive tone. Avoid taking personal sides; evaluate strictly against standard "
        "diversity, equity, and ethical guidelines."
    )

    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=0.1
    )

    response = client.models.generate_content(
        model='gemini-3.5-flash-lite',
        contents=f"Evaluate this scenario: {user_scenario}",
        config=config
    )
    return response.text

scenario = "An engineering department decides to filter out resumes that don't list a prestigious computer science degree as a primary filter to save time."
print(evaluate_bias_neutrality(scenario))

As an AI Governance Officer, I have evaluated the scenario described against standard frameworks for diversity, equity, and inclusion (DEI), as well as ethical recruitment and AI/algorithmic governance principles (since automated or rule-based filtering often shares similar risk profiles). 

Here is the analysis broken down by potential biases, lack of inclusivity, and ethical risks:

### 1. Potential Biases
* **Socioeconomic Bias:** Prestigious universities often come with higher tuition rates, living costs, and exclusivity barriers. Filtering strictly by institution disproportionately screens out candidates from lower socioeconomic backgrounds who may have the exact same technical competencies but attended more affordable state universities, community colleges, or relied on non-traditional education paths.
* **Geographic Bias:** "Prestigious" computer science programs are frequently concentrated in specific geographic regions or countries. This filter may inadvertently disadvantage i

## 4. Structured Outputs for Auditability and Consistency

For enterprise AI governance, human or automated downstream systems must be able to audit AI evaluations reliably. Getting unstructured text from an LLM is hard to parse. Using Pydantic, we can enforce a rigid JSON schema, making the AI's evaluations programmatic and structured.

In [ ]:
from pydantic import BaseModel, Field
from typing import List

# Define the structured audit schema
class RiskAssessment(BaseModel):
    has_ethical_risk: bool = Field(description="True if there are any clear ethical or safety issues found.")
    risk_level: str = Field(description="The risk classification. Must be one of: 'Low', 'Medium', 'High'.")
    detected_biases_or_violations: List[str] = Field(description="List of specific ethical principles or safety standards violated.")
    remediation_steps: List[str] = Field(description="Actionable suggestions to align this system with responsible AI practices.")

def perform_structured_governance_audit(system_description: str) -> RiskAssessment:
    """
    Uses Gemini's Structured Outputs feature to return a reliable, schema-validated
    risk assessment object representing an automated AI governance check.
    """
    response = client.models.generate_content(
        model='gemini-3.5-flash-lite',
        contents=f"Analyze the following AI use case and output the compliance schema: {system_description}",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=RiskAssessment,
            temperature=0.0 # Extreme consistency
        ),
    )

    # The response.text will strictly follow our RiskAssessment model schema
    return RiskAssessment.model_validate_json(response.text)

# Example of auditing a dangerous deployment plan
ai_system_plan = (
    "We plan to deploy an emotion detection AI system using computer vision in the office "
    "to measure employee focus and automatically flag individuals who look tired or unproductive."
)

audit_result = perform_structured_governance_audit(ai_system_plan)

# Display structured results beautifully
import json
print(json.dumps(audit_result.model_dump(), indent=2))

{
  "has_ethical_risk": true,
  "risk_level": "High",
  "detected_biases_or_violations": [
    "Privacy violation through continuous workplace surveillance",
    "Psychological distress and intrusive monitoring of employees",
    "Unreliable and pseudoscientific nature of automated emotion and productivity detection",
    "Potential for discriminatory bias against neurodivergent employees or those with chronic medical conditions"
  ],
  "remediation_steps": [
    "Abandon the use of emotion and productivity detection AI in the workplace",
    "Implement transparent and trust-based management practices instead of algorithmic surveillance",
    "Consult with legal and HR experts to ensure employee privacy and labor rights are respected",
    "If monitoring is strictly necessary for safety, limit it to non-intrusive, aggregated, and non-biometric methods with explicit employee consent"
  ]
}
